# 02. Arrêts TC — Construction des couches d'arrêts

Construction des couches d'arrêts de transport en commun structurées pour le
projet MRN : la situation actuelle `arrets_2026` à partir du GTFS ATOUMOD, puis
le réseau cible `arrets_SERM` par overlay additif des haltes projetées
(ferroviaire post-LNPN, car express).

**Prérequis** : exécuter `00_referentiels.ipynb` au préalable  
**Entrées** :
- `raw/atoumod-gtfs_20260512/` GTFS ATOUMOD Normandie
- `data/perimetre_MRN.gpkg` périmètre administratif MRN
- `data/Arrets_SERM.xlsx` haltes du réseau cible géolocalisées manuellement (provisoire)

**Sorties** : `data/arrets_2026.gpkg`, `data/arrets_SERM.gpkg`

> Filtre spatial sur le **périmètre offre** = MRN + tampon de 3 750 m (15 min
> à vélo), pour capter les arrêts hors MRN plus proches d'un habitant qu'un
> arrêt interne. Même logique offre/demande qu'au notebook 01 (cf. cadrage).

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

Détection de la racine du projet via le fichier sentinelle `.projectroot`,
puis ancrage de tous les chemins dessus. Cette approche est indépendante du
répertoire de travail : elle fonctionne sous JupyterLab comme sous VSCode,
quel que soit le réglage `notebookFileRoot`, et après un clone comme après un
téléchargement ZIP du dépôt.

In [ ]:
def trouver_racine(marqueur=".projectroot"):
    """Remonte depuis le cwd jusqu'au dossier contenant `marqueur`."""
    base = Path.cwd().resolve()
    for parent in (base, *base.parents):
        if (parent / marqueur).exists():
            return parent
    raise FileNotFoundError(f"Racine introuvable (marqueur {marqueur!r})")


ROOT = trouver_racine()
DATA_DIR = ROOT / "data"
ROW_DIR = ROOT / "raw"

## 1. Imports et paramètres

Chargement des bibliothèques, définition des constantes du projet et lecture du périmètre administratif de la MRN, qui servira, augmenté d'un tampon, à filtrer les arrêts du GTFS ATOUMOD Normandie (périmètre offre).

In [ ]:

GTFS_DIR   = Path(ROW_DIR / "atoumod-gtfs_20260512")
OUTPUT     = Path(DATA_DIR / "arrets_2026.gpkg")
LAYER_NAME = "arrets_2026"

mrn = gpd.read_file(DATA_DIR / "perimetre_MRN.gpkg")

## 2. Lecture des fichiers GTFS

Chargement des quatre tables nécessaires depuis le dossier GTFS ATOUMOD :
`routes.txt`, `trips.txt`, `stop_times.txt`, `stops.txt`.
La jointure entre ces tables suit la chaîne : `stops → stop_times → trips → routes`.

In [ ]:
routes     = pd.read_csv(GTFS_DIR / "routes.txt",
                         usecols=["route_id", "route_short_name", "route_type"])
trips      = pd.read_csv(GTFS_DIR / "trips.txt",
                         usecols=["trip_id", "route_id"])
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt",
                         usecols=["trip_id", "stop_id"], low_memory=False)
stops      = pd.read_csv(GTFS_DIR / "stops.txt")

print(f"routes     : {len(routes):>6} lignes")
print(f"trips      : {len(trips):>6} lignes")
print(f"stop_times : {len(stop_times):>6} lignes")
print(f"stops      : {len(stops):>6} lignes")

## 3. Filtre spatial des arrêts (périmètre offre)


In [ ]:
# Tous les stops → GeoDataFrame
stops["operateur"] = stops["stop_id"].str.split(":").str[-1]
stops_gdf = gpd.GeoDataFrame(
    stops,
    geometry=gpd.points_from_xy(stops["stop_lon"], stops["stop_lat"]),
    crs="EPSG:4326"
).to_crs("EPSG:2154")

# Filtre spatial : périmètre offre = MRN + tampon de 3 750 m (15 min à vélo).
# Capte les arrêts hors MRN plus proches d'un habitant qu'un arrêt interne.
# Buffer appliqué en EPSG:2154 (mètres). La population (périmètre demande)
# restera, elle, découpée à la MRN stricte (notebook 04).
TAMPON_ARRETS_M = 3750
zone_offre = mrn.to_crs("EPSG:2154").union_all().buffer(TAMPON_ARRETS_M)
stops_offre = stops_gdf[stops_gdf.within(zone_offre)]
print(f"Arrêts dans le périmètre offre (MRN + 3,75 km) : {len(stops_offre)}")

## 4. Jointure et attribution du mode

Jointure vectorisée `stops → stop_times → trips → routes` sur les arrêts MRN.
Le mode est attribué par `np.select` sur `route_type` et `route_short_name`.

| Critère                                                | Mode  |
|--------------------------------------------------------|-------|
| `route_type = 1`                                       | Métro |
| `route_type = 2`                                       | Train |
| `route_type = 3` et `route_short_name` dans TEOR_LINES | TEOR  |
| `route_type = 3` autres                                | Bus   |
| `route_type = 4`                                       | Bac   |

Résultat : un index `(stop_id × mode)` avec la liste des lignes
desservant chaque arrêt pour chaque mode.

In [ ]:
import numpy as np

TEOR_LINES = {"T1", "T2", "T3", "T4", "T5"}

# Attribution du mode par ligne
conditions = [
    routes["route_type"] == 1,
    routes["route_type"] == 2,
    (routes["route_type"] == 3) & (routes["route_short_name"].isin(TEOR_LINES)),
    (routes["route_type"] == 3) & (~routes["route_short_name"].isin(TEOR_LINES)),
    routes["route_type"] == 4,
]
choix = ["Métro", "Train", "TEOR", "Bus", "Bac"]
routes["mode"] = np.select(conditions, choix, default="Inconnu")

# Jointure stop_times → trips → routes
merged = (stop_times
    .merge(trips, on="trip_id")
    .merge(routes[["route_id", "route_short_name", "mode"]], on="route_id")
)

# Index (stop_id × mode) → liste des lignes, restreint au périmètre offre
idx = (merged[merged["stop_id"].isin(stops_offre["stop_id"])]
    [["stop_id", "mode", "route_short_name"]]
    .drop_duplicates()
    .groupby(["stop_id", "mode"])["route_short_name"]
    .agg(lambda x: ", ".join(sorted(x)))
    .reset_index()
    .rename(columns={"route_short_name": "lignes"})
)
print(f"Combinaisons (arrêt × mode) dans le périmètre offre : {len(idx)}")

## 5. Arrêts sans passage

Identification des arrêts MRN absents de `stop_times` leur mode
ne peut pas être déterminé par la jointure GTFS.

In [ ]:
agency = pd.read_csv(GTFS_DIR / "agency.txt")
agency["code"] = agency["agency_id"].str.split(":").str[0]

sans_passage = stops_offre[~stops_offre["stop_id"].isin(idx["stop_id"])]

sans_df = (sans_passage
    .groupby("operateur")
    .size()
    .reset_index(name="n_arrets")
    .merge(agency[["code", "agency_name"]],
           left_on="operateur", right_on="code", how="left")
    .sort_values("n_arrets", ascending=False)
)
print(f"Arrêts sans passage dans stop_times : {len(sans_passage)}")
print(sans_df[["operateur", "agency_name", "n_arrets"]].to_string(index=False))

# Garde-fou : avec le tampon, des arrêts hors ATOUMOD001 peuvent entrer dans
# la couronne. L'attribution en bloc du mode Bus (étape 7) suppose qu'ils sont
# tous des arrêts urbains Astuce ; on le vérifie ici.
autres = sans_df[sans_df["operateur"] != "ATOUMOD001"]
if len(autres):
    n = int(autres["n_arrets"].sum())
    print(f"\nAttention : {n} arrêt(s) sans passage hors ATOUMOD001 dans la "
          f"couronne tampon - réexaminer leur mode avant l'attribution Bus.")

## 6. Examen des arrêts sans passage

Les arrêts sans passage dans `stop_times` (nombre affiché à l'étape 5) ne
peuvent pas recevoir de mode par la jointure GTFS. Sur le périmètre MRN + tampon de 3 750 m,
ils appartiennent tous à ATOUMOD001 (Astuce).

On les visualise (en rouge) avec les arrêts Métro (en jaune) et TEOR (en bleu).

In [ ]:
import folium

sans_wgs = sans_passage.to_crs("EPSG:4326")
metro_wgs = gpd.GeoDataFrame(
    idx[idx["mode"] == "Métro"].merge(stops_offre, on="stop_id"),
    geometry="geometry", crs="EPSG:2154"
).to_crs("EPSG:4326")
teor_wgs = gpd.GeoDataFrame(
    idx[idx["mode"] == "TEOR"].merge(stops_offre, on="stop_id"),
    geometry="geometry", crs="EPSG:2154"
).to_crs("EPSG:4326")

m_sans = folium.Map(
    location=[49.44, 1.09],
    zoom_start=11,
    tiles="CartoDB positron"
)

# Arrêts sans passage (rouge)
fg_sans = folium.FeatureGroup(name="Sans passage (496)")
for _, r in sans_wgs.iterrows():
    folium.CircleMarker(
        location=[r.geometry.y, r.geometry.x],
        radius=4,
        color="#e74c3c",
        fill=True,
        fill_opacity=0.8,
        tooltip=f"{r['stop_name']} - {r['stop_id']}",
    ).add_to(fg_sans)
fg_sans.add_to(m_sans)

# Arrêts Métro (jaune)
fg_metro = folium.FeatureGroup(name="Métro")
for _, r in metro_wgs.iterrows():
    folium.CircleMarker(
        location=[r.geometry.y, r.geometry.x],
        radius=4,
        color="#1f77b4",
        fill=True,
        fill_opacity=0.8,
        tooltip=f"{r['stop_name']} - Métro - {r['lignes']}",
    ).add_to(fg_metro)
fg_metro.add_to(m_sans)

# Arrêts TEOR (bleu)
fg_teor = folium.FeatureGroup(name="TEOR")
for _, r in teor_wgs.iterrows():
    folium.CircleMarker(
        location=[r.geometry.y, r.geometry.x],
        radius=4,
        color="#f1c40f",
        fill=True,
        fill_opacity=0.8,
        tooltip=f"{r['stop_name']} - TEOR - {r['lignes']}",
    ).add_to(fg_teor)
fg_teor.add_to(m_sans)

folium.LayerControl().add_to(m_sans)
m_sans.save("../data/arrets_sans_passage.html")
print(f"✓ Carte sauvegardée ({len(sans_wgs)} arrêts sans passage, "
      f"{len(metro_wgs)} Métro, {len(teor_wgs)} TEOR)")

## 7. Traitement des arrêts sans passage

Sur le périmètre MRN + tampon de 3 750 m, les arrêts sans passage appartiennent tous à
ATOUMOD001 (Astuce) et sont distincts des arrêts Métro et TEOR d'Astuce (tous
présents dans `stop_times`) : ils reçoivent donc le mode `Bus`.

Le GeoDataFrame peut être généré.

In [ ]:
# Arrêts avec passages
avec = stops_offre.merge(idx, on="stop_id", how="inner").assign(horizon="2026")
avec = avec.rename(columns={"stop_name": "nom", "stop_id": "id_source"})
avec = avec[["nom", "horizon", "mode", "lignes", "operateur", "id_source", "geometry"]]

# Arrêts sans passage — mode Bus par défaut (valide si tous Astuce ;
# cf. étapes 6-7 et l'avertissement de l'étape 5)
sans = sans_passage.assign(horizon="2026", mode="Bus", lignes="")
sans = sans.rename(columns={"stop_name": "nom", "stop_id": "id_source"})
sans = sans[["nom", "horizon", "mode", "lignes", "operateur", "id_source", "geometry"]]

# Concaténation
gdf = gpd.GeoDataFrame(pd.concat([avec, sans], ignore_index=True), crs="EPSG:2154")
print(f"Lignes totales (arrêts × modes) : {len(gdf)}")
print(f"  dont arrêts avec passages     : {len(avec)}")
print(f"  dont arrêts sans passage      : {len(sans)}")

## 8. Contrôle qualité

Vérification de la répartition des arrêts par mode et par opérateur.

In [ ]:
assert (gdf["mode"] != "Inconnu").all(), "Mode(s) Inconnu(s) — route_type non géré dans le périmètre"

print("Répartition des arrêts par mode :")
display(gdf["mode"].value_counts().to_frame())

print("\nArrêts Métro :")
display(gdf[gdf["mode"] == "Métro"][["nom", "lignes", "operateur"]].head(10))

print("\nArrêts TEOR :")
display(gdf[gdf["mode"] == "TEOR"][["nom", "lignes", "operateur"]].head(10))

print("\nArrêts sans lignes :")
display(gdf[gdf["lignes"] == ""][["nom", "mode", "operateur"]].head(10))

print("\nRépartition par opérateur :")
display(gdf["operateur"].value_counts().to_frame())

print(f"\nCRS : {gdf.crs}")
gdf.head(5)

> Les doublons par sens (ex. *Georges Braque ×2*) sont normaux : chaque quai
> directionnel est un `stop_id` distinct dans le GTFS (aller / retour).
> Cette granularité est utile pour les calculs d'isochrones.

## 9. Export

Export de la couche en **GeoPackage** (`arrets_2026.gpkg`, couche `arrets_2026`)
dans le dossier `data/`, aux côtés des graphes OSM piéton et cyclable.

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
gdf.to_file(OUTPUT, driver="GPKG", layer=LAYER_NAME)
print(f"✓ Exporté : {OUTPUT}  ({len(gdf)} entités)")

La couche `arrets_2026.gpkg` est prête ; `arrets_SERM.gpkg` est construite
ci-dessous. Étapes suivantes :

- **Notebook 03** isochrones et table du meilleur niveau atteignable par nœud (par horizon)
- **Notebook 04** couche de population localisée (Filosofi 200 m)
- **Notebook 05** accessibilité par carreau (`niveau_actuel`, `niveau_cible`)
- **Notebook 06** comparaison avant / après réseau cible SERM

## 10. Réseau cible SERM — overlay additif

Construction de `arrets_SERM.gpkg` par **overlay additif** sur la couche 2026 :
les haltes du réseau cible (ferroviaire post-LNPN, car express) sont **ajoutées**
comme nouveaux points, sans modification des arrêts existants. La couche
résultante porte `horizon = "SERM"` pour piloter le nommage des sorties en nb03.

**Source** : `data/Arrets_SERM.xlsx`, haltes géolocalisées manuellement à partir
de la délibération du Conseil métropolitain du 15/12/2025 et des schémas de
préfiguration. Données **provisoires** (localisations en partie incertaines,
conflits de source en cours de résolution) ; leur validation relève d'une étape
ultérieure. Le mode `Car express` s'insère dans la hiérarchie entre `Métro` et
`Train` ; il est absent de l'horizon 2026.

> Nécessite `openpyxl` pour la lecture du classeur (`pip install openpyxl`).

In [ ]:
SERM_SOURCE = DATA_DIR / "Arrets_SERM.xlsx"   # source manuelle provisoire
MODES_SERM  = {"Train", "Car express"}

serm = pd.read_excel(SERM_SOURCE)
serm.columns = serm.columns.str.strip()

nom_serm  = serm["Nom"].astype(str).str.strip()
mode_serm = serm["Mode"].astype(str).str.strip()

# Mise au schéma de la couche d'arrêts (strictement identique à arrets_2026)
serm_gdf = gpd.GeoDataFrame(
    pd.DataFrame({
        "nom":       nom_serm,
        "horizon":   "SERM",
        "mode":      mode_serm,
        "lignes":    "",                                  # haltes projetées : pas de ligne GTFS
        "operateur": "SERM",                              # provenance : halte projetée
        "id_source": "SERM:" + mode_serm + ":" + nom_serm,
    }),
    geometry=gpd.points_from_xy(
        pd.to_numeric(serm["Lon WGS84"]),
        pd.to_numeric(serm["Lat WGS84"]),
    ),
    crs="EPSG:4326",
).to_crs("EPSG:2154")

print(f"Haltes cibles lues : {len(serm_gdf)}")
print(serm_gdf["mode"].value_counts().to_string())

Contrôles avant overlay. Les modes doivent appartenir à l'ensemble attendu ; les
coordonnées doivent tomber dans le périmètre offre (MRN + 3,75 km, le même qu'en
2026). Une halte hors périmètre traduit le plus souvent une erreur de saisie
(inversion lat/lon, signe) plutôt qu'une desserte réelle : l'`assert` porte sur
une boîte englobante large (garde-fou catastrophe), l'avertissement informatif
sur le périmètre offre exact. A inspecter sans bloquer, vu le caractère
provisoire des localisations.

In [ ]:
# 1) Modes attendus
modes_inattendus = set(serm_gdf["mode"]) - MODES_SERM
assert not modes_inattendus, f"Mode(s) SERM inattendu(s) : {modes_inattendus}"

# 2) Garde-fou « catastrophe de saisie » : boîte englobante large de la MRN (+20 km)
xmin, ymin, xmax, ymax = mrn.to_crs("EPSG:2154").total_bounds
MARGE_BBOX = 20_000
dans_bbox = (serm_gdf.geometry.x.between(xmin - MARGE_BBOX, xmax + MARGE_BBOX)
             & serm_gdf.geometry.y.between(ymin - MARGE_BBOX, ymax + MARGE_BBOX))
assert dans_bbox.all(), (
    "Halte(s) hors boîte englobante MRN — vérifier lat/lon :\n"
    + serm_gdf.loc[~dans_bbox, "nom"].to_string(index=False)
)

# 3) Avertissement informatif : haltes hors périmètre offre exact (MRN + 3,75 km)
hors_offre = ~serm_gdf.within(zone_offre)
if hors_offre.any():
    print(f"\u26a0 {hors_offre.sum()} halte(s) hors périmètre offre (à vérifier) :")
    print(serm_gdf.loc[hors_offre, ["nom", "mode"]].to_string(index=False))
else:
    print("Toutes les haltes cibles sont dans le périmètre offre.")

Overlay additif : la couche 2026 — ré-étiquetée `horizon = "SERM"` sans autre
modification — augmentée des nouvelles haltes. Aucun arrêt existant n'est muté ;
seul le label d'horizon, qui identifie le run, passe à `SERM`. Le schéma de
colonnes reste strictement celui de `arrets_2026`, condition de la concaténation
et de la lecture homogène en nb03.

In [ ]:
COLS = ["nom", "horizon", "mode", "lignes", "operateur", "id_source", "geometry"]

base_serm = gdf.copy()
base_serm["horizon"] = "SERM"

arrets_serm = gpd.GeoDataFrame(
    pd.concat([base_serm[COLS], serm_gdf[COLS]], ignore_index=True),
    geometry="geometry", crs="EPSG:2154",
)

# Contrôles : schéma identique à 2026, aucun mode Inconnu, comptage de l'overlay
assert list(arrets_serm.columns) == COLS, "Schéma divergent de arrets_2026"
assert (arrets_serm["mode"] != "Inconnu").all(), "Mode(s) Inconnu(s)"
assert len(arrets_serm) == len(gdf) + len(serm_gdf), "Comptage overlay incohérent"

print(f"arrets_SERM : {len(arrets_serm)} entités "
      f"({len(gdf)} repris de 2026 + {len(serm_gdf)} haltes cibles)")
print("\nRépartition par mode :")
print(arrets_serm["mode"].value_counts().to_string())

OUTPUT_SERM = DATA_DIR / "arrets_SERM.gpkg"
arrets_serm.to_file(OUTPUT_SERM, driver="GPKG", layer="arrets_SERM")
print(f"\n\u2713 Exporté : {OUTPUT_SERM}  ({len(arrets_serm)} entités)")

## 11. Visualisation de contrôle et portfolio

Carte interactive Folium des arrêts par mode.
Chaque couche est activable/désactivable via le contrôle de légende.
La carte est sauvegardée en HTML dans `data/` pour consultation portfolio hors Jupyter.

In [ ]:
import folium

# Reprojection en WGS84 pour Folium
gdf_wgs  = gdf.to_crs("EPSG:4326")
serm_wgs = serm_gdf.to_crs("EPSG:4326")        # haltes du réseau cible (nouvelles)

# --- 1. Palette accessible (Bang Wong, 2011 — sûre pour les daltonismes) ---
couleurs = {
    "Métro":       "#56B4E9",  # bleu ciel
    "TEOR":        "#E69F00",  # orange
    "Train":       "#D55E00",  # vermillon
    "Bus":         "#009E73",  # vert bleuté
    "Bac":         "#CC79A7",  # mauve
    "Car express": "#0072B2",  # bleu (mode propre au réseau cible)
}

# --- 2. Fond plus lisible que positron (voirie / libellés visibles) ---
m = folium.Map(
    location=[49.40, 1.09],
    zoom_start=11,
    tiles="CartoDB Voyager",
    control_scale=True,
)

# --- 3. Limites de la MRN (contour externe, communes dissoutes) ---
mrn_wgs = mrn.to_crs("EPSG:4326")
limite_mrn = mrn_wgs.union_all()          # fusionne les communes en un seul contour
fg_limite = folium.FeatureGroup(name="Limite MRN")
folium.GeoJson(
    data=gpd.GeoSeries([limite_mrn], crs="EPSG:4326"),
    style_function=lambda _: {
        "color": "#333333",
        "weight": 2.5,
        "dashArray": "5, 5",
        "fill": False,
    },
).add_to(fg_limite)
fg_limite.add_to(m)

# --- 4. Arrêts 2026 par mode (pastilles pleines) ---
for mode, groupe in gdf_wgs.groupby("mode"):
    fg = folium.FeatureGroup(name=mode)
    for _, r in groupe.iterrows():
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            radius=4,
            color="#ffffff",     # détourage fin liseré blanc
            weight=0.5,
            fill=True,
            fill_color=couleurs.get(mode, "#333333"),
            fill_opacity=0.9,
            tooltip=f"{r['nom']} — {r['mode']}" + (f" — {r['lignes']}" if r['lignes'] else ""),
        ).add_to(fg)
    fg.add_to(m)

# --- 5. Haltes du réseau cible SERM (anneaux : couleur = mode, anneau = projeté) ---
libelle_couche = {"Train": "Train_SERM", "Car express": "Car_express_SERM"}
for mode, groupe in serm_wgs.groupby("mode"):
    fg = folium.FeatureGroup(name=libelle_couche.get(mode, f"{mode}_SERM"))
    for _, r in groupe.iterrows():
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            radius=7,
            color=couleurs.get(mode, "#333333"),   # anneau coloré épais
            weight=2.5,
            fill=True,
            fill_color="#ffffff",                    # cœur blanc = halte projetée
            fill_opacity=0.85,
            tooltip=f"{r['nom']} — {r['mode']} — réseau cible (projeté)",
        ).add_to(fg)
    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

titre = """
<div style="position: fixed; top: 12px; left: 50%; transform: translateX(-50%);
            z-index: 1000; background: white; padding: 6px 14px;
            border: 1px solid #999; border-radius: 6px;
            font: 600 15px/1.3 sans-serif;">
  Arrêts TC — situation 2026 et réseau cible SERM
  <div style="font-weight: 400; font-size: 11px; color: #555;">
    Sources : GTFS ATOUMOD (11/06/2026) ; haltes cibles géolocalisées (provisoire)
  </div>
</div>
"""
m.get_root().html.add_child(folium.Element(titre))

m.save("../data/arrets_2026_carte.html")
print(f"✓ Carte sauvegardée — 2026 : {len(gdf_wgs)} arrêts ; "
      f"réseau cible : {len(serm_wgs)} haltes "
      f"({(serm_wgs['mode'] == 'Train').sum()} Train, "
      f"{(serm_wgs['mode'] == 'Car express').sum()} Car express)")